In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import csv
import pandas as pd
import sqlite3

db_path = 'data/Data Engineer_ETL Assignment.db'

query = """
  SELECT
    S.CUSTOMER_ID,
    I.ITEM_NAME,
    CAST(SUM(QUANTITY) AS INTEGER) AS TOTAL
  FROM (SELECT CUSTOMER_ID, AGE FROM CUSTOMERS WHERE  18 <= AGE  AND  AGE <= 35) C
  LEFT JOIN SALES S
  ON S.CUSTOMER_ID  = C.CUSTOMER_ID
  LEFT JOIN ORDERS O
  ON O.SALES_ID = S.SALES_ID
  LEFT JOIN ITEMS I
  ON I.ITEM_ID = O.ITEM_ID
  GROUP BY S.CUSTOMER_ID, I.ITEM_NAME
  HAVING SUM(QUANTITY) > 0
"""


In [3]:
#### python and sql implementation

try:
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()
  cursor.execute(query)

  rows = cursor.fetchall()
  columns = [description[0] for description in cursor.description]
  output_csv = 'ITEM_PURCAHSES_BY_CUSTOMER-SQL.csv'

  with open(f"output/{output_csv}", mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(columns)
    writer.writerows(rows)

  print(f"Query results saved to '{output_csv}'")

except Exception as e:
  print("An error occurred:", e)

finally:
  cursor.close()
  conn.close()

Query results saved to 'ITEM_PURCAHSES_BY_CUSTOMER-SQL.csv'


In [4]:
#### pandas implementation

try:
  conn = sqlite3.connect(db_path)

  customers_df = pd.read_sql_query("SELECT CUSTOMER_ID, AGE FROM CUSTOMERS WHERE 18 <= AGE AND AGE <= 35", conn)
  sales_df = pd.read_sql_query("SELECT * FROM SALES", conn)
  orders_df = pd.read_sql_query("SELECT * FROM ORDERS", conn)
  items_df = pd.read_sql_query("SELECT * FROM ITEMS", conn)

  # print("CUSTOMERS:", customers_df.columns.tolist())
  # print("SALES:", sales_df.columns.tolist())
  # print("ORDERS:", orders_df.columns.tolist())
  # print("ITEMS:", items_df.columns.tolist())

  sales_f = customers_df.merge(sales_df, on='customer_id', how='left')

  orders_f = sales_f.merge(orders_df, on='sales_id', how='left')

  full_data = orders_f.merge(items_df, on='item_id', how='left')

  result = (
      full_data
      .groupby(['customer_id', 'item_name'], as_index=False)
      .agg({'quantity': 'sum'})
  )

  result = result[result['quantity'].fillna(0) > 0]

  result['quantity'] = result['quantity'].astype(int)

  result = result.sort_values(by=['customer_id', 'item_name'])

  output_csv = "ITEM_PURCAHSES_BY_CUSTOMER-PANDAS.csv"

  result.to_csv(f"output/{output_csv}", sep=';', index=False)
  print(f"Query results saved to '{output_csv}'")

except Exception as e:
  print("An error occurred:", e)
finally:
  conn.close()

Query results saved to 'ITEM_PURCAHSES_BY_CUSTOMER-PANDAS.csv'
